In [1]:
import os, yaml, sys
import torch
from einops import reduce, rearrange
import matplotlib.pyplot as plt
import numpy as np
ENV = os.getenv("MY_ENV", "dev")
with open("../../config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
sys.path.append(paths["useful_stuff_path"])
# from useful_stuff.image_processing.utils import 
from useful_stuff.image_processing.computational_models import imgANN, load_model, get_relevant_output_layers, get_activation
from useful_stuff.general_utils.utils import get_device, get_module_by_path
device = get_device()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
from dataclasses import dataclass, field

@dataclass
class Cfg:
    model_name = "dino_v3_l";
    pkg = 'hf'
    input_size = 384
    pooling = 'all'
    model_url = "facebook/dinov3-vitl16-pretrain-lvd1689m"
cfg = Cfg()

In [3]:
# from typing import Literal
# from torchvision.models.feature_extraction import create_feature_extractor
# from transformers import AutoModel



# class imgANN():
#     def __init__(
#         self, 
#         model_name: str, 
#         pkg: str, 
#         img_size: int, 
#         relevant_layers=None, 
#         pooling='all', 
#         weights_type: str = 'DEFAULT',
#         dtype=torch.float16,
#         attn_implementation: str = None,
#         hf_class=None,
#         repo_url: str = None,
#         revision: str = None,
#         trust_remote_code: bool = False,
#     ):
#         self.model_name = model_name
#         self.pkg = pkg
#         self.img_size = img_size
#         self.pooling = pooling
#         self.weights_type = weights_type
#         self.dtype = dtype
#         self.attn_implementation = attn_implementation
#         self.hf_class = hf_class
#         self.repo_url = repo_url
#         self.revision = revision
#         self.trust_remote_code = trust_remote_code

#         self.device = get_device(verbose=True)
#         self.model = load_model(
#             model_name,
#             pkg,
#             self.device,
#             img_size=img_size,
#             weights_type=weights_type,
#             dtype=dtype,
#             attn_implementation=attn_implementation,
#             hf_class=hf_class,
#             repo_url=repo_url,
#             revision=revision,
#             trust_remote_code=trust_remote_code,
#         )
#         self.relevant_layers = get_relevant_output_layers(model_name, pkg=pkg) if relevant_layers is None else relevant_layers
#         self.features = {}
#         self.handles = {}
#     # EOF

#     def __repr__(self):
#         return f"ANN(model={self.model_name}, pkg={self.pkg}, pooling={self.pooling}, device={self.device}, img_size={self.img_size})"
#     # EOF

#     # --- GETTERS ---
#     def get_model_name(self) -> str:
#         return self.model_name

#     def get_pkg(self) -> str:
#         return self.pkg

#     def get_img_size(self) -> int:
#         return self.img_size

#     def get_relevant_layers(self) -> list[str]:
#         return self.relevant_layers

#     def get_pooling(self) -> str:
#         return self.pooling

#     def get_device(self):
#         return self.device

#     def get_model(self):       
#         return self.model

#     def get_feature_extractor(self):
#         return getattr(self, "feature_extractor", None)

#     def get_feature_extractor_layers(self):
#         return getattr(self, "feature_extractor_layers", None)

#     def get_features(self):
#         return self.features

#     def get_handles(self):
#         return self.handles

#     # --- SETTERS ---
#     def _reload_model(self):
#         self.model = load_model(
#             self.model_name,
#             self.pkg,
#             self.device,
#             img_size=self.img_size,
#             weights_type=self.weights_type,
#             dtype=self.dtype,
#             attn_implementation=self.attn_implementation,
#             hf_class=self.hf_class,
#             repo_url=self.repo_url,
#             revision=self.revision,
#             trust_remote_code=self.trust_remote_code,
#         )
#         if hasattr(self, "feature_extractor"):
#             delattr(self, "feature_extractor")
#             delattr(self, "feature_extractor_layers")
#         self.clear_hooks()

#     def set_model_name(self, model_name: str, reload_model: bool = True):
#         self.model_name = model_name
#         if reload_model:
#             self._reload_model()
#             self.relevant_layers = get_relevant_output_layers(self.model_name, pkg=self.pkg)
#         return self

#     def set_pkg(self, pkg: str, reload_model: bool = True):
#         self.pkg = pkg
#         if reload_model:
#             self._reload_model()
#             self.relevant_layers = get_relevant_output_layers(self.model_name, pkg=self.pkg)
#         return self

#     def set_img_size(self, img_size: int, reload_model: bool = True):
#         self.img_size = img_size
#         if reload_model:
#             self._reload_model()
#         return self

#     def set_relevant_layers(self, relevant_layers: list[str]):
#         self.relevant_layers = relevant_layers
#         return self

#     def set_pooling(self, pooling: str):
#         self.pooling = pooling
#         return self

#     def set_device(self, new_device):
#         self.device = new_device
#         self.model = self.model.to(self.device)
#         if hasattr(self, "feature_extractor"):
#             self.feature_extractor = self.feature_extractor.to(self.device)
#         return self

#     def set_model(self, model):
#         self.model = model.to(self.device)
#         if hasattr(self, "feature_extractor"):
#             delattr(self, "feature_extractor")
#             delattr(self, "feature_extractor_layers")
#         self.clear_hooks()
#         return self

#     # --- OTHER METHODS ---
#     def get_layer_output_shape(self, layer_name, imsize=224):
#         self.create_feature_extractor([layer_name])
#         with torch.no_grad():
#             in_proxy = torch.randn(1, 3, imsize, imsize).to(self.device)
#             tmp_shape = self.feature_extractor(in_proxy)[layer_name].shape[1:]
#         self.feature_extractor = None
#         return tmp_shape
#     # EOF 

#     def create_feature_extractor(self, layer_names: list[str] = None):
#         if layer_names is None:
#             layer_names = self.relevant_layers
#         feature_extractor = create_feature_extractor(
#             self.model, return_nodes=layer_names
#         ).to(self.device) 
#         self.feature_extractor = feature_extractor
#         self.feature_extractor_layers = layer_names
#         return feature_extractor
#     # EOF

#     def create_forward_hook(self, layer_names: list[str] = None):
#         self.clear_hooks()
#         if layer_names is None:
#             layer_names = self.relevant_layers
#         features = {}
#         handles = {}
#         for l in layer_names:
#             module = get_module_by_path(self.model, l)
#             h = module.register_forward_hook(get_activation(l, features, self.pooling))
#             handles[l] = h
#         self.handles = handles
#         self.features = features
#         return features, handles
#     # EOF

#     def extract_features(self, x, method="hook"):
#         kwargs = x if isinstance(x, dict) else {"x": x}
#         with torch.no_grad():
#             if method == "hook":
#                 self.model(**kwargs)
#                 return self.features
#             elif method == "fx":
#                 return self.feature_extractor(**kwargs)
#             else:
#                 raise ValueError(f"{method=} is not supported")
#     # EOF

#     def forward(self, x):
#         if hasattr(self, "feature_extractor"):
#             return self.feature_extractor(x)
#         return self.model(x)
#     # EOF

#     def clear_hooks(self):
#         if hasattr(self, "handles"):
#             for h in self.handles.values():
#                 h.remove()
#         self.handles = {}
#     # EOF
# # EOC

In [4]:

# m = load_model("dino_v3_l", "hf", device, hf_class=None, repo_url=cfg.model_url, attn_implementation="sdpa")

In [15]:
m = imgANN(cfg.model_name, cfg.pkg,  cfg.input_size, dtype=torch.float32, attn_implementation='sdpa', repo_url=cfg.model_url)
m.create_forward_hook()

20:07:46 - device being used: mps


({},
 {'layer.0.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x162c9bd20>,
  'layer.1.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x162c9bcb0>,
  'layer.2.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x3a827a210>,
  'layer.3.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x364d57a70>,
  'layer.4.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x364d579b0>,
  'layer.5.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x364d57a10>,
  'layer.6.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x364d57b90>,
  'layer.7.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x364d57b30>,
  'layer.8.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x364d57bf0>,
  'layer.9.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x177cf7a70>,
  'layer.10.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x162cc3830>,
  'layer.11.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x162cc35f0>,
  'layer.12.mlp.down_proj': <torch.utils.hook

In [16]:
from torchvision import transforms
from PIL import Image

# Load a sample image and preprocess it for AlexNet
import urllib.request

# Download a sample image
url = "https://www.python.org/static/community_logos/python-logo.png"
urllib.request.urlretrieve(url, "sample_image.jpg")
image = Image.open("sample_image.jpg")

# Define preprocessing for AlexNet (ImageNet normalization)
preprocess = transforms.Compose([
    transforms.Resize((cfg.input_size, cfg.input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Preprocess and add batch dimension
input_tensor = preprocess(image).unsqueeze(0).to(device)

# Forward pass
with torch.no_grad():
    output = m.model(input_tensor)



# Check captured features
for layer_name, feature in m.features.items():
    print(f"{layer_name}: {feature.shape}")

layer.0.mlp.down_proj: torch.Size([1, 594944])
layer.1.mlp.down_proj: torch.Size([1, 594944])
layer.2.mlp.down_proj: torch.Size([1, 594944])
layer.3.mlp.down_proj: torch.Size([1, 594944])
layer.4.mlp.down_proj: torch.Size([1, 594944])
layer.5.mlp.down_proj: torch.Size([1, 594944])
layer.6.mlp.down_proj: torch.Size([1, 594944])
layer.7.mlp.down_proj: torch.Size([1, 594944])
layer.8.mlp.down_proj: torch.Size([1, 594944])
layer.9.mlp.down_proj: torch.Size([1, 594944])
layer.10.mlp.down_proj: torch.Size([1, 594944])
layer.11.mlp.down_proj: torch.Size([1, 594944])
layer.12.mlp.down_proj: torch.Size([1, 594944])
layer.13.mlp.down_proj: torch.Size([1, 594944])
layer.14.mlp.down_proj: torch.Size([1, 594944])
layer.15.mlp.down_proj: torch.Size([1, 594944])
layer.16.mlp.down_proj: torch.Size([1, 594944])
layer.17.mlp.down_proj: torch.Size([1, 594944])
layer.18.mlp.down_proj: torch.Size([1, 594944])
layer.19.mlp.down_proj: torch.Size([1, 594944])
layer.20.mlp.down_proj: torch.Size([1, 594944])
la

In [17]:
np.all(np.isnan(m.features['layer.0.mlp.down_proj'].cpu().numpy()))

np.False_

In [18]:
# m.extract_features(input_tensor, 'fx')
m.extract_features({"pixel_values": input_tensor}, 'hook')

{'layer.0.mlp.down_proj': tensor([[ 0.1669, -0.1894,  0.0714,  ...,  0.1561,  0.3949,  0.2974]],
        device='mps:0'),
 'layer.1.mlp.down_proj': tensor([[-0.0892, -0.2544, -0.2863,  ...,  0.0023,  0.3708, -0.2370]],
        device='mps:0'),
 'layer.2.mlp.down_proj': tensor([[ 1.6360, -0.1276, -0.2136,  ..., -0.2487, -0.1820, -0.1170]],
        device='mps:0'),
 'layer.3.mlp.down_proj': tensor([[ 0.0123,  0.6931, -0.2591,  ...,  0.1461,  0.1938, -0.1808]],
        device='mps:0'),
 'layer.4.mlp.down_proj': tensor([[-2.8594,  0.2214,  0.0284,  ...,  0.1527,  0.2227,  0.9867]],
        device='mps:0'),
 'layer.5.mlp.down_proj': tensor([[ 0.3159, -0.7101,  0.4601,  ...,  0.3273,  0.8558, -0.7711]],
        device='mps:0'),
 'layer.6.mlp.down_proj': tensor([[-3.6887,  0.3204,  0.1809,  ..., -0.5268, -0.0352, -0.9461]],
        device='mps:0'),
 'layer.7.mlp.down_proj': tensor([[ 0.9709, -0.4414,  0.0221,  ..., -1.4238, -0.5149, -1.7967]],
        device='mps:0'),
 'layer.8.mlp.down_proj'